# ⚡ VatioPlan: Mover el consumo flexible a las horas más baratas de la energía
**Herramientas de Inteligencia Artificial, Universidad de los Andes, proyecto final 2026**

VatioPlan tiene **dos tipos de usuario**, ambos con un caso real de referencia:

| Usuario | Caso de referencia | Qué se mueve de horario |
|---|---|---|
| 🏭 Fábrica manufacturera | Cementos Argos, planta Rioclaro (Antioquia) | Molinos de crudo, cemento y carbón; trituradora |
| 🚌 Flota eléctrica | Patio Fontibón I (Refugio), SITP Bogotá (120 buses BYD) | La carga nocturna de los buses |

Este notebook recorre **todas las fases del proyecto**, en el mismo orden de la rúbrica:

| # | Fase | Herramienta |
|---|------|-------------|
| 1 | Problema | No aplica |
| 2 | Información de entrada (descarga automática) | API SIMEM/XM, NOAA |
| 3 | Valor de los datos (dato → resultado → *insight*) | pandas |
| 4 | Machine Learning: pronóstico del precio | scikit-learn |
| 5 | Dos usuarios: reglas de ingeniería + optimización | PuLP (programación entera y lineal) |
| 6 | Valor de la propuesta: ahorro con precios reales (*backtest*) | No aplica |
| 7 | IA generativa: informe para cada usuario | Gemini / Groq (gratis) |
| 8 | Orquestación y tablero | Prefect / GitHub Actions / n8n + Streamlit |

> **Todo es gratuito.** Funciona en Google Colab o en Jupyter local.

## 0. Preparación

In [ ]:
# En Google Colab clona el repositorio; en Jupyter local trabaja desde la raíz del proyecto.
import os, sys
try:
    import google.colab  # noqa
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

REPO = "https://github.com/tomasjuca1-sudo/VatioPlan.git"   # ← cámbienlo si el repositorio es otro
if EN_COLAB and not os.path.exists("VatioPlan"):
    !git clone -q {REPO}
if EN_COLAB:
    %cd VatioPlan
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")   # trabajar desde la raíz del proyecto
sys.path.insert(0, os.getcwd())
print("Carpeta de trabajo:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (12, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})

from vatioplan.datos import cargar_datos
from vatioplan.modelo import evaluar, entrenar, pronosticar, importancia_variables
from vatioplan.flotas import precios_ventana
from vatioplan.perfiles import (cargar_perfil, ventanas, planear_periodo, backtest, ahorro_backtest,
                                comparar_candidatos, resumen_para_ia, etiquetas_horas)
from vatioplan.pipeline import paso_pronostico, dias_a_planear
from vatioplan.informe import generar_informe

## 1. El problema

En Colombia la energía se compra en el **mercado mayorista**, donde el **precio de bolsa cambia cada hora**.
Cerca del 70 % de la generación es hidráulica: el precio sube con la sequía (El Niño) y todos los días
tiene un **pico nocturno (18 a 21 h)**, cuando la demanda es máxima.

Las plantas industriales programan sus equipos **sin mirar el precio**. Muchas cargas se podrían mover
de horario sin afectar la producción: hornos de secado, molinos, chillers con almacenamiento de frío,
bombeo a tanques, carga de montacargas.

**Pregunta:** *¿a qué hora debería funcionar cada equipo para producir lo mismo pagando menos energía?*

**Usuario:** jefe de planta / programador de producción. **Cliente:** empresas industriales cuyo costo
de energía depende del precio de bolsa (usuarios no regulados o con contratos indexados).

## 2. Información de entrada: descarga automática

| Fuente | Qué aporta | Acceso |
|---|---|---|
| **SIMEM (XM)**, conjunto `EC6945` | Precio de bolsa nacional **horario** | API pública GET, sin clave |
| **API XM** `PorcVoluUtilDiar`, `AporEner` | Nivel de embalses y aportes hídricos (diario) | API pública POST, sin clave |
| **NOAA CPC** | Índice ONI (El Niño / La Niña), mensual | Archivo de texto público |
| Festivos de Colombia | Calendario | Librería `holidays` |

La función `cargar_datos` descarga todo, se queda con la **versión más reciente** de cada precio
(XM revisa los precios varias veces) y guarda una **caché** en `data/` para solo bajar lo nuevo.

> Si no hay internet, genera datos **SIMULADOS** para poder probar el código (quedan marcados).

In [ ]:
df = cargar_datos(inicio="2023-01-01")
FUENTE = df.attrs["fuente"]
print("\nFuente de datos:", FUENTE)
df.tail()

### Levantamiento de los datos (tabla del anteproyecto, calculada con los datos reales)

In [ ]:
ultimo = df["precio"].last_valid_index()
tabla = pd.DataFrame([
    {"Fuente": "Precio de bolsa (SIMEM)", "Existe": "✓", "Acceso": "✓ API pública",
     "Registros": f"{df['precio'].notna().sum():,} horas",
     "Faltantes": f"{df['precio'].isna().mean():.2%}",
     "Periodo": f"{df.index.min():%Y-%m-%d} → {ultimo:%Y-%m-%d}", "Automatizable": "✓"},
] + [
    {"Fuente": nombre, "Existe": "✓" if col in df else "✗", "Acceso": "✓" if col in df else "✗",
     "Registros": f"{df[col].notna().sum() // 24:,} días" if col in df else "sin dato",
     "Faltantes": f"{df[col].isna().mean():.2%}" if col in df else "sin dato",
     "Periodo": "diario" if col in df else "sin dato", "Automatizable": "✓" if col in df else "✗"}
    for nombre, col in [("Embalses % (XM)", "vol_util_pct"), ("Aportes hídricos (XM)", "aportes_gwh"),
                        ("ONI El Niño (NOAA)", "oni")]
])
print(f"Retraso de publicación: el último precio es de hace {(pd.Timestamp.now() - ultimo).days} días")
tabla

## 3. Valor de los datos: dato → resultado → *insight*
El objetivo no es solo procesar datos sino encontrar patrones que ayuden a **decidir**.

In [ ]:
p = df["precio"].dropna()
ult = p[p.index >= p.index.max() - pd.Timedelta(days=365)]    # último año

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
perfil = ult.groupby([ult.index.hour]).agg(["mean", lambda s: s.quantile(.1), lambda s: s.quantile(.9)])
perfil.columns = ["media", "p10", "p90"]
ax[0].fill_between(perfil.index, perfil.p10, perfil.p90, alpha=.25, color="#d9480f", label="10 a 90 %")
ax[0].plot(perfil.index, perfil.media, color="#d9480f", lw=2.5, label="Promedio")
ax[0].set(title="Precio por hora del día (último año)", xlabel="Hora", ylabel="COP/kWh", xticks=range(0, 24, 2))
ax[0].legend()
nombres = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
sem = ult.groupby(ult.index.dayofweek).mean()
ax[1].bar(nombres, sem.values, color=["#868e96"] * 6 + ["#2f9e44"])
ax[1].set(title="Precio promedio por día de la semana", ylabel="COP/kWh")
plt.tight_layout(); plt.show()

diario = ult.resample("D").agg(["min", "max"])
ratio = (diario["max"] / diario["min"]).median()
barata, cara = perfil.media.idxmin(), perfil.media.idxmax()
print(f"DATO: en un día típico la hora más cara cuesta {ratio:.1f} veces la más barata.")
print(f"RESULTADO: la hora más cara en promedio es las {cara}:00 y la más barata las {barata}:00 "
      f"({perfil.media[cara] / perfil.media[barata] - 1:.0%} más cara).")
print("INSIGHT: mover consumo flexible fuera de las 18 a 21 h reduce costos SIN producir menos.")

In [ ]:
# Hidrología y clima: ¿por qué hay semanas enteras caras?
if "vol_util_pct" in df:
    d = df.resample("W").mean(numeric_only=True).dropna(subset=["precio"])
    fig, ax1 = plt.subplots(figsize=(14, 4))
    ax1.plot(d.index, d["precio"], color="#d9480f", lw=2, label="Precio semanal")
    ax1.set_ylabel("COP/kWh", color="#d9480f")
    ax2 = ax1.twinx()
    ax2.plot(d.index, d["vol_util_pct"], color="#1971c2", lw=2, label="Embalses (%)")
    ax2.set_ylabel("Volumen útil embalses (%)", color="#1971c2"); ax2.grid(False)
    plt.title("Cuando los embalses bajan, el precio sube"); plt.show()
    print(f"Correlación semanal precio y embalses: {d['precio'].corr(d['vol_util_pct']):.2f}")
    print("INSIGHT: la hidrología anticipa semanas caras → conviene adelantar producción de inventario.")

## 4. Machine Learning: pronóstico del precio
**Reto:** el precio se publica con ~3 días de retraso y queremos planear varios días adelante.
Por eso el modelo solo usa información de **hace 7 días o más** (sin fuga de datos).

**Cuidado con las correlaciones engañosas:** la demanda o la generación del mismo momento se
correlacionan con el precio, pero se deciden en el mismo despacho; usarlas sería "hacer trampa".

- **Modelo principal:** `HistGradientBoostingRegressor` (gradient boosting, scikit-learn) que **corrige el promedio
  de la misma hora en las últimas 4 semanas** según la hora, el día de la semana y los festivos.
  El promedio trae el nivel del régimen hidrológico actual; el gradient boosting aprende los ajustes de calendario.
- **Por qué así:** el nivel del precio depende del régimen del año (octubre costó 1.529 COP/kWh en 2024 y 171 en 2025).
  Un modelo que pronostica el precio absoluto aprende niveles de años pasados y falla cuando cambia el régimen.
- **Candidatos de comparación:** ingenuo (hace 7 días), promedio 4 semanas, y las versiones anteriores del GB.
- **Criterio de selección:** el **ahorro real en backtest**, no el MAE. A la planta le importa ordenar bien las
  horas de cada día, no acertar el valor exacto.
- **Prueba:** entrenar con todo menos los **últimos 90 días** y evaluar en ellos.

In [ ]:
ev = evaluar(df, dias_prueba=90)
m = ev["metricas"]
pd.DataFrame({
    "Modelo ML": [m["mae_modelo"], m["mape_modelo"], m["acierto_horas_baratas_modelo"] * 100],
    "Ingenuo (hace 7 días)": [m["mae_base"], m["mape_base"], m["acierto_horas_baratas_base"] * 100],
}, index=["MAE (COP/kWh)", "MAPE (%)", "Horas baratas acertadas (%)"]).round(1)

In [ ]:
v = pd.DataFrame({"Real": ev["real"], "Modelo ML": ev["pred"], "Ingenuo": ev["base"]}).tail(24 * 14)
ax = v.plot(color=["#212529", "#d9480f", "#adb5bd"], lw=1.6, figsize=(14, 4))
ax.set(title="Últimas 2 semanas del periodo de prueba", ylabel="COP/kWh")
plt.show()
print(f"MAE del modelo: {m['mae_modelo']:,.0f} COP/kWh (ingenuo: {m['mae_base']:,.0f}).")
print(f"Identifica {m['acierto_horas_baratas_modelo']:.0%} de las 8 horas más baratas de cada día "
      f"(ingenuo: {m['acierto_horas_baratas_base']:.0%}). Para decidir entre modelos se usa el ahorro real "
      "en backtest (sección 6).")

In [ ]:
imp = importancia_variables(ev["modelo"], df)
imp.head(10)[::-1].plot.barh(color="#1971c2", figsize=(8, 4), title="¿Qué variables usa el modelo?")
plt.xlabel("Aumento del error si se desordena la variable (COP/kWh)"); plt.show()

### Pronóstico de los próximos días (modelo re-entrenado con todos los datos)

In [ ]:
modelo = entrenar(df)
pron = paso_pronostico(df, modelo, dias_plan=3)
hist = df["precio"].tail(24 * 10)
plt.plot(hist.index, hist.values, color="#212529", label="Real (último publicado)")
plt.plot(pron.index, pron.values, color="#d9480f", ls="--", lw=2, label="Pronóstico (días a planear)")
plt.title("Precio de bolsa: historia reciente y pronóstico"); plt.ylabel("COP/kWh"); plt.legend(); plt.show()

## 5. Dos tipos de usuario: reglas de ingeniería + optimización
Los dos usuarios usan **el mismo pronóstico**. Cambian las reglas y lo que se mueve de horario.
Cada caso está en un archivo de `config/` con sus fuentes y supuestos.

**🏭 Fábrica (Argos Rioclaro)**: programación entera (PuLP + CBC). Para cada día decide a qué horas opera cada equipo:

| Regla | Significado |
|---|---|
| R1 | Cada equipo cumple sus horas diarias (no se sacrifica producción) |
| R2 | Solo opera en su ventana permitida (p. ej. la trituradora depende del turno de cantera) |
| R3 | Los equipos continuos operan en un solo bloque |
| R4 | La potencia total no supera la potencia contratada |
| R5 | Corrida mínima: un molino no se prende y apaga cada hora |

**🚌 Flota (patio Fontibón I, SITP)**: programación lineal. Decide cuántos kW recibe cada grupo de buses en cada hora,
en una ventana de 12:00 a 12:00 para que la noche quede completa:

| Regla | Significado |
|---|---|
| R1 | Cada grupo sale con la carga pedida |
| R2 | Solo carga mientras está en el patio |
| R3 | No supera la potencia de sus cargadores |
| R4 | La suma no supera el límite del patio |

In [ ]:
fabrica = cargar_perfil("fabrica")
flota = cargar_perfil("flota")
for p in [fabrica, flota]:
    print(p.etiqueta, ":", p.nombre)
    print("Supuestos:", p.supuestos, "\n")
display(fabrica.tabla_elementos())
display(flota.tabla_elementos())

In [ ]:
pron_completo = pronosticar(df, modelo)
fechas = dias_a_planear(df, pron_completo)

def graficar(perfil):
    fecha = ventanas(pron_completo, perfil, fechas)[0]
    precios = precios_ventana(pron_completo, fecha, perfil.hora_inicio)
    ref, opt = perfil.plan_referencia(), perfil.optimizar(precios)
    x = etiquetas_horas(perfil)
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
    for ax, (titulo, plan) in zip(axes, [(perfil.nombre_referencia, ref), ("VatioPlan", opt)]):
        kw = perfil.potencia_kw(plan)
        abajo = np.zeros(24)
        for col in kw.columns:
            ax.bar(range(24), kw[col].values, bottom=abajo, label=col,
                   color="#ced4da" if col == "Carga base" else None)
            abajo += kw[col].values
        ax.axhline(perfil.limite_kw, color="red", ls="--", lw=1)
        ax2 = ax.twinx(); ax2.plot(range(24), precios, color="black", lw=2); ax2.set_ylabel("COP/kWh")
        ax.set(title=f"{titulo}, costo ${perfil.costo(plan, precios):,.0f}", xlabel="Hora", ylabel="kW")
        ax.set_xticks(range(0, 24, 3)); ax.set_xticklabels(x[::3])
    axes[0].legend(fontsize=7, loc="upper left")
    plt.suptitle(f"{perfil.etiqueta.split(' ', 1)[1]}, ventana desde {fecha} {perfil.hora_inicio:02d}:00"); plt.show()
    return opt

opt_fabrica = graficar(fabrica)

In [ ]:
fabrica.validar(opt_fabrica)[["regla", "elemento", "resultado"]]

In [ ]:
opt_flota = graficar(flota)
flota.validar(opt_flota)[["regla", "elemento", "resultado"]]

## 6. Valor de la propuesta: ¿cuánto se habría ahorrado de verdad? (*backtest*)
Durante los 90 días de prueba, cada día se planeó **con el pronóstico** y el costo se calculó **con el precio real**.
Se compara contra el plan de referencia (operación actual de la fábrica o carga inmediata de la flota) y contra el
"plan perfecto" (si se conociera el precio de antemano). La tabla compara todos los candidatos del modelo.

In [ ]:
resultados = {}
for perfil in [fabrica, flota]:
    bt = backtest(ev["real"], ev["pred"], perfil)
    real, maximo = ahorro_backtest(bt)
    anual = (bt["costo_actual"] - bt["costo_con_pronostico"]).sum() / len(bt) * 365
    print(f"{perfil.etiqueta}: ahorro real {real:.1f}% (máximo {maximo:.1f}%) → ≈ ${anual:,.0f} COP al año "
          f"con exposición a bolsa de {perfil.exposicion_bolsa:.0%}")
    ax = (bt["costo_actual"] - bt["costo_con_pronostico"]).cumsum().div(1e6).plot(
        lw=2.5, figsize=(12, 3.5), label="Con pronóstico (precio real)")
    (bt["costo_actual"] - bt["costo_perfecto"]).cumsum().div(1e6).plot(ax=ax, color="#adb5bd", ls="--",
                                                                       label="Máximo teórico")
    ax.set(title=f"Ahorro acumulado, {perfil.nombre}", ylabel="Millones de COP"); ax.legend(); plt.show()
    comparacion = comparar_candidatos(ev, bt, perfil)
    display(comparacion.round(1))
    resultados[perfil.tipo] = {"bt": bt, "comparacion": comparacion}

## 7. IA generativa: un informe para cada usuario (jefe de planta y jefe de patio)
El LLM recibe **solo los números calculados** (JSON) y se le prohíbe inventar cifras: el modelo y las
reglas deciden; la IA **redacta y prioriza**.

Opcional, API key gratuita:
- Gemini: https://aistudio.google.com/apikey → `GEMINI_API_KEY`
- Groq: https://console.groq.com/keys → `GROQ_API_KEY`

Sin key se usa una plantilla (el pipeline nunca se detiene).

In [ ]:
# os.environ["GEMINI_API_KEY"] = "pega-aqui-tu-key"     # ← opcional
resumenes = {}
for perfil in [fabrica, flota]:
    plan = planear_periodo(pron_completo, perfil, ventanas(pron_completo, perfil, fechas))
    resumenes[perfil.tipo] = resumen_para_ia(plan, pron_completo, ev, resultados[perfil.tipo]["bt"], perfil, FUENTE)
print(json.dumps(resumenes["flota"], ensure_ascii=False, indent=1)[:1500], "...")

In [ ]:
from IPython.display import Markdown, display
for tipo, resumen in resumenes.items():
    texto, proveedor = generar_informe(resumen)
    print("Generado con:", proveedor)
    display(Markdown(texto))

## 8. Orquestación y tablero

**Orquestación (todos los días ~6:00 a. m.)**, después de que XM actualiza (~4:00 a. m.):

```
Descargar datos → Entrenar/pronosticar → Por usuario: optimizar + reglas + backtest → Informe IA → Publicar/Notificar
```
- **GitHub Actions** (`.github/workflows/vatioplan_diario.yml`): corre el pipeline en la nube, gratis, y guarda
  `data/` y `salidas/` en el repositorio. Así el tablero arranca con datos reales aunque SIMEM bloquee.
- **Prefect** (`flujo_prefect.py`): cada paso es una tarea con reintentos; si un plan viola una regla, no se publica.
- **n8n** (`n8n_workflow.json`): lanza el workflow de GitHub, espera y envía un informe al jefe de planta y otro al jefe de patio.

**Tablero low-code (Streamlit):** `streamlit run app.py`: selector de tipo de usuario, plan del día, caso real,
pronóstico, validación y el informe IA con los parámetros de la barra lateral.

In [ ]:
# Ejecutar el pipeline completo, tal como lo haría el orquestador:
from vatioplan.pipeline import ejecutar
salida = ejecutar(verbose=False)
for raiz, _, archivos in sorted(os.walk("salidas")):
    print(raiz, sorted(archivos))

## 9. Conclusiones (complétenlas con las cifras de arriba)
1. **Problema:** el precio de bolsa cambia cada hora y sube con la sequía; la industria y las flotas eléctricas
   consumen por costumbre, sin mirarlo.
2. **Datos:** públicos, horarios, gratuitos y automatizables (SIMEM/XM + NOAA).
3. **ML:** el gradient boosting sobre el promedio de 4 semanas se eligió por ahorro real en backtest. El hallazgo
   clave es que el ancla (promedio reciente) captura el régimen del mercado; los modelos que pronostican el precio
   absoluto fallan cuando cambia el régimen.
4. **Reglas:** el plan siempre cumple las reglas de cada usuario (validado automáticamente).
5. **Valor:** ahorro medido con precios reales para dos usuarios reales de referencia (Argos Rioclaro y el patio
   Fontibón I), sin producir menos y sin que un bus salga sin carga.
6. **Innovación:** la misma herramienta sirve a fábricas y a flotas eléctricas, con un archivo de configuración por cliente.